## Veri Ön İşleme

Bu aşamada veri seti modelleme için hazırlanacaktır. Eksik değerler ele alınacak, kategorik değişkenler dönüştürülecek ve eğitim-test verileri ayrılacaktır.

In [12]:
import pandas as pd
import numpy as np

from pathlib import Path

data_path = Path("../data/raw")
file_path = data_path / "E Commerce Dataset.xlsx"

df = pd.read_excel(file_path, sheet_name="E Comm")

df.shape

(5630, 20)

In [13]:
category_mapping = {
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone"
    },
    "PreferredPaymentMode": {
        "CC": "Credit Card",
        "COD": "Cash on Delivery"
    },
    "PreferedOrderCat": {
        "Mobile": "Mobile Phone"
    }
}

for column, mapping in category_mapping.items():
    df[column] = df[column].replace(mapping)

## Hedef Değişkenin ve Özelliklerin Ayrılması

Churn değişkeni tahmin edilmek istenen hedef olarak belirlenmiştir. CustomerID ise yalnızca müşteri kimliği olduğu için modelin özelliklerinden çıkarılmıştır.

In [14]:
X = df.drop(columns=["Churn", "CustomerID"])
y = df["Churn"]

print("Özellik sayısı:", X.shape[1])
print("Hedef dağılımı:")
print(y.value_counts(normalize=True))

Özellik sayısı: 18
Hedef dağılımı:
Churn
0    0.831616
1    0.168384
Name: proportion, dtype: float64


## Eğitim ve Test Verilerinin Ayrılması

Veri seti %80 eğitim ve %20 test olacak şekilde ayrılmıştır. Stratify parametresi kullanılarak her iki veri setinde de churn oranının korunması sağlanmıştır.

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Eğitim seti:", X_train.shape)
print("Test seti:", X_test.shape)

Eğitim seti: (4504, 18)
Test seti: (1126, 18)


## Eksik Değerlerin Doldurulması ve Değişkenlerin Dönüştürülmesi

Sayısal değişkenlerde eksik değerler medyan ile doldurulmuş ve ölçeklendirilmiştir. Kategorik değişkenlerde eksik değerler en sık görülen değer ile doldurulmuş ve One-Hot Encoding uygulanmıştır.

Dönüşümler yalnızca eğitim verisi üzerinde öğrenilerek test verisine uygulanmıştır.

In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_features = X_train.select_dtypes(
    include=["object", "str", "category"]
).columns

In [17]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    )),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="most_frequent"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

In [18]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Eğitim verisi:", X_train_processed.shape)
print("Test verisi:", X_test_processed.shape)

Eğitim verisi: (4504, 37)
Test verisi: (1126, 37)


# Modelleme

Bu aşamada müşteri kaybını tahmin etmek için Logistic Regression, Random Forest ve XGBoost modelleri kullanılmıştır. Modeller aynı eğitim ve test verileri üzerinde eğitilerek performansları karşılaştırılmıştır.

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
}

In [21]:
trained_models = {}

for name, model in models.items():
    model.fit(X_train_processed, y_train)
    trained_models[name] = model

    print(f"{name} eğitildi.")

Logistic Regression eğitildi.
Random Forest eğitildi.
XGBoost eğitildi.


In [22]:
import joblib
from pathlib import Path

models_path = Path("../models")
models_path.mkdir(parents=True, exist_ok=True)

for name, model in trained_models.items():
    file_name = name.lower().replace(" ", "_") + ".joblib"
    file_path = models_path / file_name

    joblib.dump(model, file_path)

    print(f"{name} kaydedildi: {file_path}")

Logistic Regression kaydedildi: ../models/logistic_regression.joblib
Random Forest kaydedildi: ../models/random_forest.joblib
XGBoost kaydedildi: ../models/xgboost.joblib
